In [ ]:
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util

# Step 1: Load dataset
file_path = "archive/student_profiles.jsonl"

data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data.append(json.loads(line.strip()))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(data)

print("Dataset Preview:")
print(df.head())


Dataset Preview:
                 Name  Age     Sex                    Major       Year   GPA  \
0       Thomas Ibarra   24    Male             Biochemistry     Senior  2.83   
1  Timothy Wilson PhD   23    Male  Business Administration     Senior  2.37   
2         Nancy Brown   23  Female              Archaeology  Sophomore  3.44   
3         Donna Reyes   19  Female   Mechanical Engineering     Senior  2.35   
4       Mary Gonzales   24  Female                Sociology   Freshman  2.42   

                           Hobbies Country State/Province  \
0      [table tennis, photography]     USA  Massachusetts   
1      [snowboarding, ice skating]  Canada        Ontario   
2            [dancing, bouldering]     USA           Iowa   
3          [astronomy, geocaching]  Canada        Ontario   
4  [puzzle solving, bird watching]  Canada        Ontario   

             Unique Quality                                              Story  
0  Wildlife conservationist  Thomas Ibarra was a senio

In [4]:
# Step 2: Clean text data (example: removing NaN, trimming spaces)
# Assume there's a 'profile' or 'description' column containing text
text_column = "profile" if "profile" in df.columns else df.columns[0]  # fallback
df[text_column] = df[text_column].astype(str).str.strip()

In [5]:
# Step 3: Load MiniLM model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

c:\Users\Vaishnavi\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vaishnavi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is n

In [6]:
# Step 4: Encode dataset
corpus = df[text_column].tolist()
corpus_embeddings = model.encode(corpus, convert_to_tensor=True, show_progress_bar=True)

Batches: 100%|██████████| 727/727 [00:46<00:00, 15.56it/s]


In [7]:
# Step 5: Function to find top 15 matches for a query
def find_best_matches(query, top_k=15):
    query_embedding = model.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=top_k)[0]
    
    results = []
    for hit in hits:
        results.append({
            "text": corpus[hit['corpus_id']],
            "score": float(hit['score'])
        })
    return results

# Example Query
query = "I am looking for a student skilled in machine learning and Python"
matches = find_best_matches(query)

print("\nTop 15 Matches:")
for idx, match in enumerate(matches, start=1):
    print(f"{idx}. {match['text']} (Score: {match['score']:.4f})")



Top 15 Matches:
1. Travis Lara PhD (Score: 0.3154)
2. Mr. Kyle Oconnor PhD (Score: 0.2912)
3. Christopher Lawson (Score: 0.2868)
4. Ms. Elizabeth Wilkerson PhD (Score: 0.2856)
5. Vanessa Pearson (Score: 0.2834)
6. Christopher Davies (Score: 0.2825)
7. Sherry Lawson (Score: 0.2823)
8. Jason Johnson PhD (Score: 0.2816)
9. Tamara Montgomery (Score: 0.2810)
10. Tamara Montgomery (Score: 0.2810)
11. Alejandra Cook PhD (Score: 0.2808)
12. Zoe Nichols (Score: 0.2785)
13. Lydia Pearson (Score: 0.2783)
14. Melissa Dodson (Score: 0.2726)
15. Leah Lowery (Score: 0.2720)
